# SQL Indexing Deep Dive — B-tree, Hash, GIN, Partial

An index is a separate data structure that Postgres maintains in parallel with the table. Reads get faster; writes get slower. The wrong index type is often worse than no index — the planner ignores it and seq scans anyway. Understanding what each index type can and cannot do is how you diagnose slow queries at scale.

## B-tree indexes — the default

Every `CREATE INDEX` without a `USING` clause creates a B-tree. It is the right choice for the vast majority of columns:

- **Balanced tree structure** — O(log N) lookup regardless of table size. A 500K-row table with a B-tree index requires ~19 comparisons to find a row. A full seq scan requires 500K reads.
- **Values stored in sorted order** — this is what enables range scans (`BETWEEN`, `>`, `<`), prefix matching (`LIKE 'prefix%'`), and `ORDER BY` without a sort step.
- **Multi-column B-tree** — `(a, b)` sorts by `a` first, then `b` within each group of identical `a` values. `WHERE a = X` uses the index. `WHERE b = X` alone cannot — `b` is not globally sorted.
- **Column order rule** — put the equality column first, the range column last: `(metric_name, recorded_at DESC)` lets `WHERE metric_name = 'cpu' AND recorded_at >= X` hit both columns in order.
- **High cardinality** — B-tree is most useful when the column has many distinct values (`endpoint_id`, `hostname`, `recorded_at`). Low-cardinality columns like `status` (`open`/`closed`) are poor candidates.

In [ ]:
# Path resolution — find _setup regardless of working directory
from pathlib import Path
import sys
for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

from db_connections import get_postgres_conn
import pandas as pd

conn = get_postgres_conn()
cur = conn.cursor()
print("Connected to PostgreSQL")

In [ ]:
# Create B-tree index on alerts.created_at — enables range + ORDER BY without sort step
cur.execute("""
    SELECT indexname FROM pg_indexes
    WHERE schemaname = 'telemetry'
      AND tablename = 'alerts'
      AND indexname = 'idx_alerts_created'
""")
exists = cur.fetchone()
if not exists:
    cur.execute("""
        CREATE INDEX idx_alerts_created
        ON telemetry.alerts (created_at DESC)
    """)
    conn.commit()
    print("Index created: idx_alerts_created")
else:
    print("Index already exists: idx_alerts_created")

# EXPLAIN ANALYZE — should show Index Scan using idx_alerts_created
cur.execute("""
    EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
    SELECT alert_id, severity, message
    FROM telemetry.alerts
    WHERE created_at >= NOW() - INTERVAL '7 days'
    ORDER BY created_at DESC
    LIMIT 20
""")
plan = cur.fetchall()
print("\nEXPLAIN ANALYZE — B-tree on alerts.created_at:")
for row in plan:
    print(row[0])
print("\nLook for: Index Scan using idx_alerts_created")

## Hash indexes — equality only

Hash indexes compute a hash of the indexed value and store `hash → page location`. Lookup is O(1) for exact equality:

- **Equality only** — supports `=` operator and nothing else. No `<`, no `>`, no `BETWEEN`, no `LIKE`, no `ORDER BY`.
- **O(1) lookup** — for pure equality at very large scale, a hash lookup beats B-tree's O(log N). The difference is meaningful at 100M+ rows.
- **Not WAL-logged before Postgres 10** — before v10, hash indexes were unsafe across crashes. Since v10 they are fully WAL-logged and safe.
- **When to use** — lookup tables with exact-match-only access patterns: session token validation, user ID lookup, UUID-based joins where you never need range queries.
- **When NOT to use** — any column where you also need sort, range, or prefix queries. A B-tree on the same column covers equality too, so Hash only wins in the narrow O(1) vs O(log N) case at extreme scale.

In [ ]:
# Create Hash index on endpoints.hostname — O(1) equality lookup
cur.execute("""
    CREATE INDEX IF NOT EXISTS idx_endpoints_hostname_hash
    ON telemetry.endpoints USING HASH (hostname)
""")
conn.commit()
print("Index created: idx_endpoints_hostname_hash (HASH)")

# Exact equality — Hash index can serve this
cur.execute("""
    EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
    SELECT * FROM telemetry.endpoints
    WHERE hostname = 'srv-00001.citi.internal'
""")
plan_eq = cur.fetchall()
print("\nEXPLAIN — exact equality (Hash can serve this):")
for row in plan_eq:
    print(row[0])

# Prefix LIKE — Hash index cannot help, planner will ignore it
cur.execute("""
    EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
    SELECT * FROM telemetry.endpoints
    WHERE hostname LIKE 'srv-000%'
""")
plan_like = cur.fetchall()
print("\nEXPLAIN — LIKE prefix scan (Hash index skipped):")
for row in plan_like:
    print(row[0])
print("\nHash index skipped for LIKE — B-tree handles prefix matching")

## GIN indexes — composite values and full-text search

GIN (Generalized Inverted Index) maps each element inside a composite value to the set of rows containing it:

- **Inverted index** — instead of `row → value`, GIN stores `token → [row1, row2, ...]`. Exactly how a search engine index works.
- **Use cases** — `tsvector` full-text search, `JSONB` containment queries (`@>`, `<@`), array operators, `pg_trgm` trigram matching.
- **Slower to build, faster for containment** — building a GIN index is more expensive than B-tree because every element inside the composite value must be indexed individually.
- **In our schema** — `alerts.message` contains human-readable text. A GIN index on `to_tsvector('english', message)` enables `@@` full-text search against the indexed token list.
- **`pg_trgm` + GIN** — with the trigram extension enabled, `LIKE '%substring%'` (leading wildcard) can use a GIN index. This is the only way to index substring search on large text columns.
- **Operator support** — `@@` (tsquery match), `@>` (contains), `<@` (contained by), `&&` (overlap). These operators cannot use B-tree.

In [ ]:
# Add generated tsvector column — stored so GIN can index it without function overhead
cur.execute("""
    ALTER TABLE telemetry.alerts
    ADD COLUMN IF NOT EXISTS message_tsv tsvector
    GENERATED ALWAYS AS (to_tsvector('english', message)) STORED
""")
conn.commit()
print("Column added: message_tsv (GENERATED ALWAYS AS tsvector STORED)")

# Create GIN index on the tsvector column
cur.execute("""
    CREATE INDEX IF NOT EXISTS idx_alerts_gin_message
    ON telemetry.alerts USING GIN (message_tsv)
""")
conn.commit()
print("Index created: idx_alerts_gin_message (GIN)")

# Full-text search using @@ operator — GIN index serves this
cur.execute("""
    EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
    SELECT alert_id, severity, message
    FROM telemetry.alerts
    WHERE message_tsv @@ to_tsquery('english', 'CPU & exceeded')
    LIMIT 10
""")
plan_gin = cur.fetchall()
print("\nEXPLAIN — GIN full-text search (look for Bitmap Index Scan on idx_alerts_gin_message):")
for row in plan_gin:
    print(row[0])

# Show actual results
cur.execute("""
    SELECT alert_id, severity, message
    FROM telemetry.alerts
    WHERE message_tsv @@ to_tsquery('english', 'CPU & exceeded')
    LIMIT 10
""")
rows = cur.fetchall()
df = pd.DataFrame(rows, columns=['alert_id', 'severity', 'message'])
print("\nFull-text search results:")
print(df.to_string(index=False))
print("\nGIN index enables full-text search — same results as ILIKE but with index support")

## Partial indexes — index only the rows you query

A partial index adds a `WHERE` clause to the index definition. Only rows satisfying that condition appear in the index:

- **Smaller index** — if only 20% of alerts are `status = 'open'`, a partial index is 20% the size of a full index. It fits in memory better and scans faster.
- **Faster writes** — fewer rows to index means lower overhead on `INSERT` and `UPDATE`.
- **Planner requirement** — the query's `WHERE` clause must logically imply the index's `WHERE` clause for the planner to use it. `WHERE severity = 'critical' AND status = 'open'` implies the partial index `WHERE severity = 'critical' AND status = 'open'`.
- **Best use cases** — any column with a skewed distribution where queries consistently target the minority: open tickets, active sessions, recent data, error-only log entries.
- **Most underused index type** — many teams index entire columns when they only ever query a small slice. Partial indexes are the fix.
- **Cannot replace a full index** — a query without the partial index condition will fall back to seq scan or a different index.

In [ ]:
# Create partial index on open critical alerts — only indexes the hot minority of rows
cur.execute("""
    CREATE INDEX IF NOT EXISTS idx_alerts_open_critical
    ON telemetry.alerts (created_at DESC)
    WHERE severity = 'critical' AND status = 'open'
""")
conn.commit()
print("Index created: idx_alerts_open_critical (partial — critical + open only)")

# Compare partial index size vs full index size
cur.execute("""
    SELECT
        ui.indexrelname AS indexname,
        pg_size_pretty(pg_relation_size(ui.indexrelid)) AS index_size
    FROM pg_stat_user_indexes ui
    WHERE ui.relname = 'alerts'
    ORDER BY pg_relation_size(ui.indexrelid) DESC
""")
df_sizes = pd.DataFrame(cur.fetchall(), columns=['indexname', 'index_size'])
print("\nIndex sizes on telemetry.alerts:")
print(df_sizes.to_string(index=False))

# Query must match the partial index WHERE clause exactly
cur.execute("""
    EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
    SELECT alert_id, endpoint_id, message, created_at
    FROM telemetry.alerts
    WHERE severity = 'critical'
      AND status = 'open'
    ORDER BY created_at DESC
    LIMIT 20
""")
plan_partial = cur.fetchall()
print("\nEXPLAIN — partial index on critical+open (look for Index Scan using idx_alerts_open_critical):")
for row in plan_partial:
    print(row[0])

## Index bloat and maintenance

Indexes are not free. They degrade over time and carry ongoing write overhead:

- **Dead index entries** — every `UPDATE` or `DELETE` leaves dead entries in indexes just like dead tuples in the heap. `VACUUM` cleans heap dead tuples but does not compact index pages.
- **Index bloat** — over time, dead entries cause the index to grow beyond what its live data would require. Bloated indexes are slower to scan and consume more memory.
- **REINDEX CONCURRENTLY** — rebuilds the index from scratch into a new structure, then swaps atomically. No table lock — reads and writes continue during the rebuild. Use after bulk loads or when bloat is confirmed.
- **`pg_stat_user_indexes`** — shows `idx_scan` (how many times this index was used), `idx_tup_read` (index entries read), `idx_tup_fetch` (heap rows actually fetched). An index with `idx_scan = 0` after weeks of normal traffic is dead weight — it slows every write with zero read benefit.
- **Unused index removal** — `DROP INDEX CONCURRENTLY idx_name` removes an index without locking. After bulk-adding indexes, check usage after a few days of traffic and drop anything unused.

In [ ]:
# Audit all indexes on the telemetry schema — usage stats + size + definition
cur.execute("""
    SELECT
        i.indexname,
        i.tablename,
        pg_size_pretty(pg_relation_size(ui.indexrelid)) AS size,
        ui.idx_scan AS times_used,
        ui.idx_tup_read AS tuples_read,
        i.indexdef
    FROM pg_indexes i
    JOIN pg_stat_user_indexes ui
        ON i.indexname = ui.indexrelname
        AND i.schemaname = ui.schemaname
    WHERE i.schemaname = 'telemetry'
    ORDER BY ui.idx_scan DESC
""")
df_idx = pd.DataFrame(
    cur.fetchall(),
    columns=['indexname', 'tablename', 'size', 'times_used', 'tuples_read', 'indexdef']
)
print("Index audit — telemetry schema:")
print(df_idx.to_string(index=False))
print("\nidx_scan=0 after normal traffic = candidate for removal")
print("REINDEX CONCURRENTLY idx_name — rebuilds without table lock")

## Index type decision guide

| Index type | Operator support | Best for | Avoid when |
|------------|-----------------|----------|------------|
| B-tree | =, <, >, BETWEEN, LIKE 'x%', ORDER BY | Most columns, ranges, sorting | Very low cardinality |
| Hash | = only | Pure equality lookups at massive scale | Ranges, sorting, LIKE |
| GIN | @>, <@, @@, array contains | Arrays, JSONB, full-text search | Simple scalar columns |
| Partial | Same as base type | Skewed distributions, hot subsets | Uniform distributions |
| BRIN | Range on physically ordered data | Append-only time-series, large tables | Random write patterns |

## Interview Q&A — what gets asked at Staff DE level

**"What is a composite index and why does column order matter?"**  
→ A B-tree composite index on `(a, b)` is sorted by `a` first, then `b` within each group of identical `a` values. `WHERE a = X` can use the index because `a` is globally sorted. `WHERE b = X` alone cannot — `b` is only locally sorted within each `a` group. Rule: put the equality column first, the range column last. Index `(metric_name, recorded_at DESC)` serves `WHERE metric_name = 'cpu' AND recorded_at >= X` perfectly.

**"When would you use a partial index?"**  
→ When queries consistently filter on a low-cardinality condition that selects a small fraction of rows. Example: `WHERE status = 'open'` on an alerts table where 70% of rows are `resolved`. The partial index is 30% the size of a full index — it fits in memory better, scans faster, and has lower write overhead. The query's `WHERE` clause must logically imply the index condition.

**"Why is my index not being used?"**  
→ Four common reasons: (1) selectivity too low — planner estimates seq scan is cheaper when many rows match; (2) column wrapped in a function — `WHERE LOWER(hostname) = 'x'` cannot use an index on `hostname`, create a functional index instead; (3) statistics are stale — run `ANALYZE tablename`; (4) composite index column order mismatch — leading column not in the `WHERE` clause.

**"What is the difference between Index Scan and Bitmap Heap Scan?"**  
→ Index Scan fetches heap pages one at a time in index order — each entry in the index causes a separate heap page fetch. Fast for small result sets but generates random I/O at larger counts. Bitmap Heap Scan first scans the index to build a bitmap of matching heap pages, then reads those pages in physical order — amortizes the random I/O. Postgres picks based on estimated row count: few rows → Index Scan, more rows → Bitmap Heap Scan, many rows → Seq Scan.

**"How do you handle `LIKE '%substring%'` with an index?"**  
→ Standard B-tree cannot handle a leading wildcard — the index is sorted by the full value, so `%x` has no useful starting point. Two options: (1) enable `pg_trgm` extension and create a GIN index — breaks text into 3-character trigrams, supports arbitrary substring matching; (2) switch to full-text search with a generated `tsvector` column and a GIN index. Never run `ILIKE '%x%'` on a large table without one of these.

**"What is index bloat and how do you fix it?"**  
→ Indexes accumulate dead entries from `UPDATE` and `DELETE` just like heap tables accumulate dead tuples. `VACUUM` cleans the heap but index pages are not compacted — they hold dead entries until the page is reused organically. On a write-heavy table this causes the index to grow beyond its live data. Check `pg_stat_user_indexes` for usage and `pgstattuple` for bloat. Fix with `REINDEX CONCURRENTLY indexname` — rebuilds from scratch without locking the table.

---
*Simplicity and clarity is Gold.*